# Chapter 3: Creating and Using a Structure Dataset

## Introduction

The structure dataset component of the PG2 dataset system allows you to work with molecular structure data from various file formats including PDB, mmCIF, and binary CIF. This chapter explains how to create, load, and use structure datasets in your projects.

## Understanding the StructureDataset

The `StructureDataset` class is designed to handle structural data of biological molecules with flexibility in the backend library used. It provides functionality for:

1. Loading structure data from various file formats (PDB, mmCIF, binary CIF)
2. Supporting multiple backend libraries (Biopython and Biotite)
3. Handling both single structure files and directories of structures
4. Providing a consistent interface regardless of the backend used

## Backend Libraries

The `StructureDataset` supports two popular Python libraries for structural bioinformatics:

1. **Biopython**: A widely used library with comprehensive tools for computational molecular biology
2. **Biotite**: A modern, object-oriented library for the analysis and processing of biological data

The system automatically selects an appropriate backend based on what's installed in your environment, with Biopython being the first choice if both are available.

In [ ]:
# Check which backend libraries are available
from importlib.util import find_spec

biopython_available = find_spec("Bio") is not None
biotite_available = find_spec("biotite") is not None

print(f"Biopython available: {biopython_available}")
print(f"Biotite available: {biotite_available}")

if not biopython_available and not biotite_available:
    print("\nNeither Biopython nor Biotite is installed.")
    print("To use the StructureDataset, install at least one of these libraries:")
    print("  pip install biopython")
    print("  pip install biotite")

## Creating a Structure Dataset

There are two main ways to create a structure dataset:

### 1. From a dataset.toml File

The most common approach is to create a structure dataset through a `dataset.toml` file:

In [ ]:
import os
from pathlib import Path

# First, let's create a simple PDB file for demonstration
# This is a minimal PDB file with just a few atoms
minimal_pdb = '''\
ATOM      1  N   ALA A   1      -0.525  -1.362   0.000  1.00  0.00           N  
ATOM      2  CA  ALA A   1       0.000   0.000   0.000  1.00  0.00           C  
ATOM      3  C   ALA A   1       1.520   0.000   0.000  1.00  0.00           C  
ATOM      4  O   ALA A   1       2.075   0.746   0.827  1.00  0.00           O  
ATOM      5  CB  ALA A   1      -0.507   0.787   1.207  1.00  0.00           C  
TER       6      ALA A   1
END
'''

# Create directory if it doesn't exist
os.makedirs("../example_data", exist_ok=True)

# Write the PDB file
pdb_path = "../example_data/sample_structure.pdb"
with open(pdb_path, "w") as f:
    f.write(minimal_pdb)

print(f"Created sample PDB file at {os.path.abspath(pdb_path)}")

# Now create a dataset.toml file that references this structure
dataset_toml_content = '''
[resources]
records = "../example_data/sample_data.csv"  # Path to a CSV file
structure = "../example_data/sample_structure.pdb"

[records]
columns = ["sequence", "score"]
sequence_feature = "sequence"

[metadata]
name = "Sample Structure Dataset"
description = "A sample dataset with structure for demonstration purposes"
'''

# Write the content to a file
with open("sample_structure_dataset.toml", "w") as f:
    f.write(dataset_toml_content)

print(f"Created sample_structure_dataset.toml at {os.path.abspath('sample_structure_dataset.toml')}")

# Create a simple CSV file for the records
import pandas as pd

data = {
    'sequence': ['ALANINE'],
    'score': [1.0]
}

df = pd.DataFrame(data)
csv_path = "../example_data/sample_data.csv"
df.to_csv(csv_path, index=False)

# Now try to load the dataset from the TOML file
from pg2_dataset.dataset import Dataset

try:
    # Load dataset from TOML file
    dataset = Dataset.from_toml("sample_structure_dataset.toml")
    
    # Access the structure dataset
    structure_dataset = dataset.structure
    
    if structure_dataset:
        print("\nSuccessfully loaded structure dataset from TOML file")
    else:
        print("\nStructure dataset not loaded. This might be because neither Biopython nor Biotite is installed.")
except Exception as e:
    print(f"\nError loading dataset: {e}")

### 2. Directly Creating a StructureDataset

You can also create a `StructureDataset` directly:

In [ ]:
try:
    from pg2_dataset.backends import StructureDataset
    
    # Create the dataset with a file path to a single structure
    structure_dataset = StructureDataset(file_path="../example_data/sample_structure.pdb")
    
    print("Successfully created structure dataset directly")
    print(f"Number of structures: {len(structure_dataset.structures)}")
except ImportError as e:
    print(f"Import error: {e}")
    print("This might be because neither Biopython nor Biotite is installed.")
except Exception as e:
    print(f"Error creating dataset: {e}")

## Supported File Formats

The `StructureDataset` supports the following file formats:

1. **PDB** (.pdb): The classic Protein Data Bank format
2. **mmCIF** (.cif): The macromolecular Crystallographic Information File format
3. **Binary CIF** (.bcif): A binary version of the CIF format

The appropriate parser is selected automatically based on the file extension.

## Structure Managers

The `StructureDataset` uses a dependency injection pattern with `StructureManager` implementations:

1. `BiopythonStructureManager`: Uses Biopython's parsers
2. `BiotiteStructureManager`: Uses Biotite's parsers

The appropriate manager is selected automatically based on available libraries in your environment.

## Accessing Structure Data

Once you've loaded a structure dataset, you can access the structures:

In [ ]:
try:
    if 'structure_dataset' in locals() and structure_dataset and structure_dataset.structures:
        # Access all loaded structures
        structures = structure_dataset.structures
        print(f"Loaded structures: {list(structures.keys())}")
        
        # Access a specific structure by its ID (usually the filename)
        structure_id = next(iter(structures.keys()))
        structure = structures[structure_id]
        print(f"\nAccessed structure with ID: {structure_id}")
        
        # Determine which backend is being used
        if biopython_available and hasattr(structure, "get_atoms"):
            print("Using Biopython backend")
            # Access model, chain, residue, and atom data
            model = structure[0]  # First model
            print(f"Number of chains: {len(model)}")
            
            for chain in model:
                print(f"Chain ID: {chain.id}")
                for residue in chain:
                    print(f"  Residue: {residue.resname} {residue.id[1]}")
                    for atom in residue:
                        print(f"    Atom: {atom.name}, Coordinates: {atom.coord}")
        
        elif biotite_available and hasattr(structure, "coord"):
            print("Using Biotite backend")
            # Access atom data
            print(f"Number of atoms: {len(structure)}")
            if hasattr(structure, "chain_id"):
                print(f"Chain IDs: {structure.chain_id}")
            if hasattr(structure, "res_name"):
                print(f"Residue names: {structure.res_name}")
            if hasattr(structure, "coord"):
                print(f"Atom coordinates shape: {structure.coord.shape}")
                print(f"First atom coordinates: {structure.coord[0]}")
        
        else:
            print("Unknown structure format or backend")
            print(f"Structure type: {type(structure)}")
    else:
        print("No structure dataset available or no structures loaded")
except Exception as e:
    print(f"Error accessing structure data: {e}")

## Loading Multiple Structures

The `StructureDataset` can load multiple structures from a directory:

In [ ]:
# Create a directory with multiple structure files
multi_struct_dir = "../example_data/structures"
os.makedirs(multi_struct_dir, exist_ok=True)

# Create a few simple PDB files
for i in range(1, 4):
    with open(f"{multi_struct_dir}/structure_{i}.pdb", "w") as f:
        # Modify the minimal PDB slightly for each file
        modified_pdb = minimal_pdb.replace("ALA A   1", f"ALA A   {i}")
        f.write(modified_pdb)

print(f"Created multiple structure files in {os.path.abspath(multi_struct_dir)}")

# Now try to load the structures from the directory
try:
    from pg2_dataset.backends import StructureDataset
    
    # Load all structures from the directory
    multi_structure_dataset = StructureDataset(file_path=multi_struct_dir)
    
    # Access the loaded structures
    print(f"\nLoaded {len(multi_structure_dataset.structures)} structures")
    for structure_id, structure in multi_structure_dataset.structures.items():
        print(f"Structure ID: {structure_id}")
except ImportError as e:
    print(f"\nImport error: {e}")
    print("This might be because neither Biopython nor Biotite is installed.")
except Exception as e:
    print(f"\nError loading multiple structures: {e}")

## Installation Requirements

To use the `StructureDataset`, you need to install either Biopython or Biotite:

In [ ]:
# This cell shows how to install the required libraries
# Uncomment and run if needed

# !pip install biopython
# !pip install biotite

## Example Usage

Here's a complete example of working with a structure dataset:

In [ ]:
from pg2_dataset.dataset import Dataset
from pg2_dataset.backends import StructureDataset

try:
    # Method 1: Load from dataset.toml
    dataset = Dataset.from_toml("sample_structure_dataset.toml")
    structure_dataset = dataset.structure
    
    if not structure_dataset:
        # Method 2: Create directly if Method 1 failed
        structure_dataset = StructureDataset(file_path="../example_data/sample_structure.pdb")
    
    # Check which structures are loaded
    print(f"Loaded structures: {list(structure_dataset.structures.keys())}")
    
    # Process the first structure
    structure_id = next(iter(structure_dataset.structures.keys()))
    structure = structure_dataset.structures[structure_id]
    
    # The rest depends on which backend is being used (Biopython or Biotite)
    try:
        # Check if it's a Biopython structure
        model = structure[0]
        print(f"Using Biopython backend")
        print(f"Structure ID: {structure_id}")
        print(f"Number of chains: {len(model)}")
        
        # Count residues and atoms
        residue_count = sum(1 for _ in model.get_residues())
        atom_count = sum(1 for _ in model.get_atoms())
        print(f"Number of residues: {residue_count}")
        print(f"Number of atoms: {atom_count}")
        
        # Calculate center of mass
        import numpy as np
        coords = np.array([atom.coord for atom in model.get_atoms()])
        center_of_mass = coords.mean(axis=0)
        print(f"Center of mass: {center_of_mass}")
        
    except (TypeError, AttributeError):
        # It's probably a Biotite structure
        print(f"Using Biotite backend")
        print(f"Structure ID: {structure_id}")
        
        # If it's an atom array
        if hasattr(structure, "coord"):
            print(f"Number of atoms: {len(structure)}")
            print(f"Coordinate shape: {structure.coord.shape}")
            
            # Calculate center of mass
            center_of_mass = structure.coord.mean(axis=0)
            print(f"Center of mass: {center_of_mass}")
except Exception as e:
    print(f"Error in example usage: {e}")

## Best Practices

1. **Backend Consistency**: For consistency in your code, consider standardizing on either Biopython or Biotite
2. **Error Handling**: Add try-except blocks when working with structures to handle potential format issues
3. **File Organization**: When working with multiple structures, use a consistent naming convention
4. **Path Configuration**: Use absolute paths or ensure relative paths are correct
5. **Memory Management**: Be aware that large structures can consume significant memory

## Troubleshooting

Common issues when working with structure datasets:

1. **Missing Dependencies**: Ensure either Biopython or Biotite is installed
2. **File Not Found**: Verify that the path to your structure file is correct
3. **Unsupported Format**: Check that your file has one of the supported extensions (.pdb, .cif, .bcif)
4. **Backend-Specific Code**: Be aware that code written for one backend may not work with the other
5. **Memory Issues**: Large structures may cause memory problems; consider processing data in chunks

## Summary

The structure dataset provides a flexible way to work with molecular structure data in the PG2 dataset system. By supporting multiple backends (Biopython and Biotite) and file formats (PDB, mmCIF, binary CIF), it allows you to choose the tools that best fit your needs. The dependency injection pattern ensures that your code can work with either backend, providing flexibility and future-proofing your analysis pipelines.

In [ ]:
# Clean up the files we created
import os
import shutil

try:
    os.remove("sample_structure_dataset.toml")
    os.remove("../example_data/sample_structure.pdb")
    os.remove("../example_data/sample_data.csv")
    shutil.rmtree("../example_data/structures")
    print("Cleaned up sample files")
except Exception as e:
    print(f"Error cleaning up: {e}")